In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np


In [ ]:
transform = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize((0.5, 0.5, 0.5),
(0.5, 0.5, 0.5))
])


In [ ]:
train_dataset = torchvision.datasets.CIFAR10(
root="./data",
train=True,
download=True,
transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
root="./data",
train=False,
download=True,
transform=transform
)


In [ ]:
train_dataset[0][0].shape

In [ ]:
len(train_dataset)

In [ ]:
X = torch.stack([p[0] for p in train_dataset])
y = torch.tensor([p[1] for p in train_dataset])

In [ ]:
test=X[0]

In [ ]:
test.shape

In [ ]:
image_size=32
patch_size=4
patches=(image_size//patch_size)**2
d_embedding=128

In [ ]:
layer=nn.Conv2d(in_channels=3,out_channels=d_embedding,kernel_size=patch_size,stride=patch_size)

In [ ]:
converted=layer(test)

In [ ]:
converted.shape

In [ ]:
train_loader=DataLoader(train_dataset,32,shuffle=True)

In [ ]:
class PatchEmbeddingLayer(nn.Module):
  def __init__(self, patch_size, in_channels, embed_dim):
    super().__init__()
    self.patch_size = patch_size
    self.in_channels = in_channels
    self.embed_dim = embed_dim
    self.patch_embedding=nn.Conv2d(in_channels=self.in_channels,out_channels=self.embed_dim,kernel_size=self.patch_size,stride=self.patch_size)

  def forward(self,x):
    x=self.patch_embedding(x)
    x=x.flatten(2)
    x=x.transpose(1,2)
    return x


In [ ]:
class CLSLayer(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim=embed_dim
    self.cls_token = nn.Parameter(torch.zeros(1,1,self.embed_dim))
  def forward(self,x):
    cls_token=self.cls_token.expand(x.shape[0],-1,-1)
    x=torch.cat((cls_token,x),dim=1)
    return x

In [ ]:
class PositionalLayer(nn.Module):
  def __init__(self,patches,embed_dim):
    super().__init__()
    self.patches=patches
    self.embed_dim=embed_dim
    self.positional_embedding=nn.Parameter(torch.zeros(1,self.patches+1,self.embed_dim))

  def forward(self,x):
    x=x+self.positional_embedding
    return x


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embed_dim, num_heads):
      super(MultiHeadAttention, self).__init__()

      self.embed_dim = embed_dim
      self.num_heads = num_heads
      self.head_dim = embed_dim // num_heads

      assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

      self.Wq = nn.Linear(embed_dim, embed_dim)
      self.Wk = nn.Linear(embed_dim, embed_dim)
      self.Wv = nn.Linear(embed_dim, embed_dim)
      self.Wo = nn.Linear(embed_dim, embed_dim)

      self.scale = self.head_dim ** -0.5

  def forward(self, x):

      B, T, C = x.shape

      Q = self.Wq(x)
      K = self.Wk(x)
      V = self.Wv(x)

      Q = Q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
      K = K.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
      V = V.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)

      attention_scores = (Q @ K.transpose(-2, -1)) * self.scale

      attention_weights = torch.softmax(attention_scores, dim=-1)

      output = attention_weights @ V

      output = output.transpose(1, 2).reshape(B, T, C)

      output = self.Wo(output)

      return output



In [ ]:
class MLP(nn.Module):
  def __init__(self,embed_dim,hidden_dim):
    super().__init__()
    self.embed_dim=embed_dim
    self.hidden_dim=hidden_dim
    self.linear1=nn.Linear(self.embed_dim,self.hidden_dim)
    self.linear2=nn.Linear(self.hidden_dim,self.embed_dim)
    self.gelu=nn.GELU()

  def forward(self,x):
    x=self.linear1(x)
    x=self.gelu(x)
    x=self.linear2(x)
    return x

In [ ]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,embed_dim,hidden_dim,num_heads):
    super().__init__()
    self.embed_dim=embed_dim
    self.hidden_dim=hidden_dim
    self.num_heads=num_heads
    self.multi_head_attention=MultiHeadAttention(embed_dim=self.embed_dim,num_heads=self.num_heads)
    self.layer_norm_1=nn.LayerNorm(self.embed_dim)
    self.layer_norm_2=nn.LayerNorm(self.embed_dim)
    self.mlp=MLP(embed_dim=self.embed_dim,hidden_dim=self.hidden_dim)
  def forward(self,x):
    x_norm=self.layer_norm_1(x)
    attention=self.multi_head_attention(x_norm)
    x=x+attention
    x_norm=self.layer_norm_2(x)
    mlp=self.mlp(x_norm)
    x=x+mlp
    return x

In [ ]:
class VisionTransformer(nn.Module):
  def __init__(self,patches,patch_size, in_channels, embed_dim,num_heads,hidden_dim,encoder_layer,num_classes):
    super().__init__()
    self.patch_size=patch_size
    self.in_channels=in_channels
    self.embed_dim=embed_dim
    self.num_heads=num_heads
    self.hidden_dim=hidden_dim
    self.patches=patches
    self.encoder_layer=encoder_layer
    self.num_classes=num_classes


    self.patch_layer=PatchEmbeddingLayer(patch_size=self.patch_size,in_channels=self.in_channels,embed_dim=self.embed_dim)
    self.cls_layer=CLSLayer(embed_dim=self.embed_dim)
    self.positional_layer=PositionalLayer(patches=self.patches,embed_dim=self.embed_dim)
    self.final_norm=nn.LayerNorm(embed_dim)
    self.classifier=nn.Linear(embed_dim,num_classes)
    self.encoder_blocks = nn.ModuleList([
    TransformerEncoderBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.num_heads
          )
          for _ in range(self.encoder_layer)
      ])
  def forward(self,x):
    x=self.patch_layer(x)
    x=self.cls_layer(x)
    x=self.positional_layer(x)

    for encoder in self.encoder_blocks:
      x=encoder(x)
    x=self.final_norm(x)
    cls_tokens=x[:,0]
    logits=self.classifier(cls_tokens)
    return logits

In [ ]:
model = VisionTransformer(
patches=64,
patch_size=4,
in_channels=3,
embed_dim=128,
num_heads=4,
hidden_dim=256,
encoder_layer=6,
num_classes=10
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
epochs = 10

for epoch in range(epochs):

  model.train()

  total_loss = 0
  correct = 0
  total = 0

  for images, labels in train_loader:

      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      outputs = model(images)

      loss = criterion(outputs, labels)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()

      _, predicted = torch.max(outputs, 1)

      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  train_acc = 100 * correct / total

  print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f} | Train Acc: {train_acc:.2f}%")

Epoch 1/10 | Loss: 1304.8473 | Train Acc: 38.64%
Epoch 2/10 | Loss: 1018.0512 | Train Acc: 52.78%
Epoch 3/10 | Loss: 908.0973 | Train Acc: 58.03%
Epoch 4/10 | Loss: 837.6689 | Train Acc: 61.69%
